In [1]:
import sys,os,re,argparse,glob
import pandas as pd
from scipy.io import loadmat
from pathlib import Path
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

import numpy as np


In [2]:
ROOT = Path('/home/tntiniak/Work/observatory_benchmark')
PHYSICELL_DIR = ROOT / 'PhysiCell' / 'results' / 'mechanics_pushing_extended_hertz'
OUTPUT_FILENAME = 'physicell_mechanics_pushing_extended_hertz.csv'


In [3]:
for folder in PHYSICELL_DIR.glob('drag_*'):
    records = []
    for xml_path in sorted(folder.glob('output*.xml')):
        root = ET.parse(xml_path).getroot()
        current_time = float(root.findtext('.//current_time'))
        mat_name = root.findtext('.//simplified_data/filename')
        if mat_name is None:
            raise ValueError(f'No cell data filename found in {xml_path}')

        cells = loadmat(folder / mat_name)['cells']
        positions = cells[1:4, :].T
        radii = cells[37, :]
        if positions.shape[0] != 2:
            raise ValueError(f'Expected 2 cells, found {positions.shape[0]} in {mat_name}')

        distance = float(np.linalg.norm(positions[0] - positions[1]))
        overlap = float(radii[0] + radii[1] - distance)
        # velocity = np.mean(cells[56, :])
        records.append({
            'time_min': current_time,
            'distance_um': distance,
            'radius_sum_um': float(radii[0] + radii[1]),
            'overlap_um': overlap,
            # 'velocity_um_per_min': velocity
        })
    frame = pd.DataFrame.from_records(records).sort_values('time_min').reset_index(drop=True)
    print(frame)
    frame.to_csv(folder / OUTPUT_FILENAME, index=False)

      time_min  distance_um  radius_sum_um  overlap_um
0          0.0     9.000000      10.000008    1.000008
1          0.1     9.101646      10.000008    0.898362
2          0.2     9.154313      10.000008    0.845695
3          0.3     9.204514      10.000008    0.795494
4          0.4     9.250281      10.000008    0.749727
...        ...          ...            ...         ...
2996     299.6     9.999913      10.000008    0.000095
2997     299.7     9.999913      10.000008    0.000095
2998     299.8     9.999913      10.000008    0.000095
2999     299.9     9.999913      10.000008    0.000095
3000     300.0     9.999913      10.000008    0.000095

[3001 rows x 4 columns]
      time_min  distance_um  radius_sum_um  overlap_um
0          0.0     9.000000      10.000008    1.000008
1          0.1     9.010165      10.000008    0.989843
2          0.2     9.016786      10.000008    0.983221
3          0.3     9.023359      10.000008    0.976648
4          0.4     9.029867      10.0000